# 06 — Branch agents raise staffing requests

**Goal (PROBLEM.md):** *"a branch that needs more staff can access it."* Step one: the branch
coordinator agent converts the federated forecast into a structured staffing request.

Pre-registered request rule (fires on **sustained** pressure, not one bad month):
- unit is **RED in ≥ 2 of the last 3 observed months**, AND
- 12-month federated forecast (own fine-tuned model, recursive rollout) minus recent
  throughput is a **positive gap**.

A request reads: *"General Surgery @ RDE needs +5.5 WTE (1,090 slots/month)"* — sized by the
gap, expressed in staff via `SLOTS_PER_WTE = 200` (stated planning assumption).

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

SLOTS_PER_WTE = 200
w = pd.read_csv("../data/processed/ops_panel.csv")
feat = pd.read_csv("../data/processed/features.csv")
saved = torch.load("../data/processed/models_src.pt", weights_only=False)
specs, FEATS = saved["specs"], saved["feats"]
spec_idx = {s: i for i, s in enumerate(specs)}
ser = ["provider_code", "specialty_code"]
months = sorted(w["month"].unique())
trusts = sorted(w["provider_code"].unique())
icb_of = w.drop_duplicates("provider_code").set_index("provider_code")["icb"]
device = "cuda" if torch.cuda.is_available() else "cpu"

class Net(nn.Module):
    def __init__(self, emb=4, hidden=32):
        super().__init__()
        self.emb = nn.Embedding(len(specs), emb)
        self.mlp = nn.Sequential(nn.Linear(len(FEATS) + emb, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, x, s):
        return x[:, 0] + self.mlp(torch.cat([x, self.emb(s)], dim=1)).squeeze(-1)

# 12-month recursive rollout with the branch's own fine-tuned federated model
@torch.no_grad()
def forecast_12m(trust, as_of):
    sub = feat[(feat["provider_code"] == trust) & (feat["month"] <= as_of)]
    hist = sub.pivot(index="month", columns="specialty_code", values="y").sort_index()
    last = sub[sub["month"] == sub["month"].max()].set_index("specialty_code")
    cols = list(hist.columns)
    mu, sd = last["mu"].reindex(cols).values, last["sd"].reindex(cols).values
    ctx = last[["backlog", "throughput"]].reindex(cols).values  # frozen standardized context
    sidx = torch.tensor([spec_idx[c] for c in cols]).to(device)
    model = Net().to(device)
    model.load_state_dict(saved["fed_ft"][trust])
    model.eval()
    buf = hist.values
    p = pd.Period(as_of, "M")
    preds = []
    for h in range(1, 13):
        tm = p + h
        lag12 = hist.loc[(tm - 12).strftime("%Y-%m")].values
        x = np.column_stack([buf[-1], buf[-2], buf[-3], lag12, ctx[:, 0], ctx[:, 1],
                             np.full(len(cols), np.sin(2 * np.pi * tm.month / 12)),
                             np.full(len(cols), np.cos(2 * np.pi * tm.month / 12))])
        y = model(torch.tensor(x, dtype=torch.float32).to(device), sidx).cpu().numpy()
        buf = np.vstack([buf, y])
        preds.append(np.expm1(y * sd + mu))
    return pd.DataFrame(np.array(preds), columns=cols)

print(f"units: {w.groupby(ser).ngroups} | latest month: {months[-1]} | models for {len(saved['fed_ft'])} branches")

In [ ]:
# the request rule: sustained RED + positive federated-forecast gap
def make_requests(as_of):
    upto = [m for m in months if m <= as_of]
    red2 = w[w["month"].isin(upto[-3:])].groupby(ser)["red"].sum() >= 2
    thr6 = w[w["month"].isin(upto[-6:])].groupby(ser)["throughput"].mean()
    snap = w[w["month"] == as_of].set_index(ser)
    rows = []
    for t in trusts:
        fc = forecast_12m(t, as_of)
        for spec in fc.columns:
            if not red2.get((t, spec), False):
                continue
            gap = fc[spec].mean() - thr6[(t, spec)]
            if gap <= 0:
                continue
            rows.append({"provider_code": t, "specialty_code": spec, "icb": icb_of[t],
                         "gap": round(gap), "wte": round(gap / SLOTS_PER_WTE, 1),
                         "share52": snap.loc[(t, spec), "share52"],
                         "backlog_now": snap.loc[(t, spec), "backlog"],
                         "priority": gap * (1 + snap.loc[(t, spec), "share52"])})
    return pd.DataFrame(rows).sort_values("priority", ascending=False).reset_index(drop=True)

requests = make_requests(months[-1])
requests.to_csv("../data/processed/requests_latest.csv", index=False)
print(f"as-of {months[-1]}: {len(requests)} staffing requests | "
      f"total ask: {requests['gap'].sum():,} slots/month = {requests['wte'].sum():,.0f} WTE")
print("\ntop requests (as the board will read them):")
name_of = pd.read_csv("../data/processed/rtt_long.csv",
                      usecols=["specialty_code", "specialty_name"]).drop_duplicates().set_index("specialty_code")["specialty_name"]
for _, r in requests.head(10).iterrows():
    print(f"  {name_of[r['specialty_code']].replace(' Service', ''):<28} @ {r['provider_code']}  ->  "
          f"+{r['wte']:.1f} WTE ({r['gap']:,} slots/mo)  [board {r['icb']}]")

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c3c2b7",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8,
    "axes.labelcolor": "#52514e", "xtick.color": "#898781", "ytick.color": "#898781",
})
top = requests.head(15).iloc[::-1]
labels = top["provider_code"] + " · " + top["specialty_code"]
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.barh(labels, top["wte"], color="#2a78d6", height=0.65)
for i, v in enumerate(top["wte"].values):
    ax.text(v + top["wte"].max() * 0.01, i, f"+{v:.1f}", va="center", fontsize=7.5, color="#0b0b0b")
ax.set_title(f"Largest staffing requests, {months[-1]} (WTE, forecast-driven)")
ax.set_xlabel("requested WTE")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()